# Step 2b — Per-Arm Scores, DeLong Test, Full Permutation

Step 2 established the result:

| evaluation | arm-level AUC |
| --- | --- |
| A random split (conventional) | 0.933 |
| **B leave-one-compound-out** | **0.832** (95% CI 0.764–0.893) |
| C dose only, LOCO | 0.580 |
| C dose only, raw | 0.690 |

Two things are still missing before submission.

### 1. The permutation p-value is at its floor
Step 2 reported `p = 0.091` from 10 permutations. With 10 permutations the smallest
reachable p is 1/11 = 0.0909, so that value means **no** permutation matched the
observed AUC — the observed statistic sits about **6.9 SD** above the null mean
(0.846 vs 0.529 ± 0.046). Reported as 0.091 a reviewer reads "not significant".
This notebook runs **100** permutations so the p-value reflects the evidence.

### 2. The transcriptome-vs-dose gap has no formal test
The +0.142 difference needs a significance test. Both scores are computed on the
same 396 arms, so the curves are correlated and an independent-samples test would be
wrong. **DeLong's test** for correlated ROC curves is the correct choice.

This notebook re-runs the LOCO pass once, saving per-arm scores this time, then runs
both. Budget 2–3 hours. The expression matrix is reused if Step 2 left it in place.

## 1. Setup and reload

In [ ]:
!pip install xgboost mygene -q

import json, gc
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

RANDOM_STATE = 42
EARLY_DAY    = 4
N_FEATURES   = 100
RULE         = "union"
N_PERM       = 100          # the fix: 10 -> 100

WORK   = Path("/content/step2"); WORK.mkdir(parents=True, exist_ok=True)
STEP1  = Path("/content/tggates_annot")
MATRIX = WORK / "affy_tggates_repeat.tsv"
ZEN    = "https://zenodo.org/records/20437047/files/"
np.random.seed(RANDOM_STATE)

if not MATRIX.exists() or MATRIX.stat().st_size < 600_000_000:
    import subprocess
    print("fetching the matrix (~666 MB) ...")
    subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", str(MATRIX),
                    ZEN + "affy_tggates_repeat.tsv?download=1"], check=True)
print(f"matrix: {MATRIX.stat().st_size/1e6:.0f} MB")

## 2. Rebuild the modelling dataset (identical to Step 2)

In [ ]:
# Step 1b wrote these. A restarted Colab session loses /content, so offer to
# re-upload step1_outputs.zip rather than failing on a missing file.
NEEDED_CSV = [f"arms_{RULE}.csv", f"array_labels_{RULE}.csv"]
missing = [f for f in NEEDED_CSV if not (STEP1 / f).exists()]
if missing:
    print("missing from", STEP1, ":", missing)
    print("upload step1_outputs.zip (from Step 1b) now")
    import zipfile
    from google.colab import files
    up = files.upload()
    STEP1.mkdir(parents=True, exist_ok=True)
    for name in up:
        with zipfile.ZipFile(name) as z:
            z.extractall(STEP1)
    still = [f for f in NEEDED_CSV if not (STEP1 / f).exists()]
    if still:
        raise FileNotFoundError(f"still missing after upload: {still}. "
                                f"Re-run Step 1b to regenerate them.")
    print("extracted to", STEP1)
else:
    print("Step 1 outputs found:", NEEDED_CSV)

arms   = pd.read_csv(STEP1 / f"arms_{RULE}.csv")
labels = pd.read_csv(STEP1 / f"array_labels_{RULE}.csv")
labels["bc"] = labels.BARCODE.astype(str).str.zfill(12)

admitted = arms[arms.status == "admitted"].copy()
adm_keys = set(zip(admitted.compound, admitted.dose_level))
outcome_map = {(c, d): int(o) for c, d, o in
               zip(admitted.compound, admitted.dose_level, admitted.outcome)}

day4 = labels[labels.day == EARLY_DAY].copy()
ms = day4[[(c, d) in adm_keys for c, d in zip(day4.COMPOUND_NAME, day4.DOSE_LEVEL)]].copy()
ms["arm"] = ms.COMPOUND_NAME + " | " + ms.DOSE_LEVEL
ms["y"]   = [outcome_map[(c, d)] for c, d in zip(ms.COMPOUND_NAME, ms.DOSE_LEVEL)]

header = pd.read_csv(MATRIX, sep="\t", nrows=0)
cols = set(header.columns); idx_col = list(header.columns)[0]
need = [b for b in sorted(set(ms.bc)) if b in cols]
expr = pd.read_csv(MATRIX, sep="\t", usecols=[idx_col] + need,
                   index_col=0, dtype={c: "float32" for c in need})
expr.index = expr.index.astype(str)
X_all = expr.T
del expr; gc.collect()
X_all = X_all[[g for g in X_all.columns if not str(g).startswith("AFFX")]]

ms = ms[ms.bc.isin(X_all.index)].copy()
X       = X_all.loc[ms.bc.values]
y       = ms.y.values
groups  = ms.COMPOUND_NAME.values
arm_id  = ms.arm.values
dose_ord = ms.DOSE_LEVEL.map({"Low": 1, "Middle": 2, "High": 3}).values

arm_truth = (pd.DataFrame({"arm": arm_id, "y": y, "compound": groups})
               .groupby("arm").agg(y=("y", "max"), compound=("compound", "first")))
print(f"X {X.shape} | arms {len(arm_truth)} | positive arms {int(arm_truth.y.sum())}")

## 3. Pipeline (identical to Step 2) plus the light variant

In [ ]:
def differential_genes(Xtr, ytr, alpha=0.05, min_lfc=0.5):
    pos, neg = Xtr.values[ytr == 1], Xtr.values[ytr == 0]
    if len(pos) < 2 or len(neg) < 2:
        return Xtr.columns[:500]
    _, p = ttest_ind(pos, neg, axis=0, equal_var=False)
    lfc = pos.mean(0) - neg.mean(0)
    ok = ~np.isnan(p); padj = np.ones_like(p)
    padj[ok] = multipletests(p[ok], method="fdr_bh")[1]
    keep = (padj < alpha) & (np.abs(lfc) >= min_lfc)
    return Xtr.columns[keep] if keep.sum() >= 20 else Xtr.columns[np.argsort(p)[:500]]

def select_features(Xtr, ytr, n=N_FEATURES):
    rf = RandomForestClassifier(n_estimators=600, class_weight="balanced_subsample",
                                random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr)
    return Xtr.columns[np.argsort(rf.feature_importances_)[::-1][:n]]

def make_model(ytr):
    pos = max(int(ytr.sum()), 1)
    return XGBClassifier(n_estimators=400, max_depth=3, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         scale_pos_weight=(len(ytr) - pos) / pos,
                         random_state=RANDOM_STATE, eval_metric="logloss")

def fit_predict(Xtr, ytr, Xte):
    deg = differential_genes(Xtr, ytr)
    feats = select_features(Xtr[deg], ytr)
    return make_model(ytr).fit(Xtr[feats], ytr).predict_proba(Xte[feats])[:, 1], feats

def fit_predict_fast(Xtr, ytr, Xte, n=40):
    _, p = ttest_ind(Xtr.values[ytr == 1], Xtr.values[ytr == 0], axis=0, equal_var=False)
    p = np.nan_to_num(p, nan=1.0)
    feats = Xtr.columns[np.argsort(p)[:n]]
    pos = max(int(ytr.sum()), 1)
    m = XGBClassifier(n_estimators=120, max_depth=3, learning_rate=0.1,
                      scale_pos_weight=(len(ytr) - pos) / pos,
                      random_state=RANDOM_STATE, eval_metric="logloss")
    return m.fit(Xtr[feats], ytr).predict_proba(Xte[feats])[:, 1]

def arm_scores(sample_proba, arms_vec):
    return (pd.DataFrame({"arm": arms_vec, "p": sample_proba})
              .groupby("arm")["p"].mean())

## 4. LOCO pass, keeping the per-arm scores this time

These scores are what the DeLong test needs, and they are worth saving as a
supplementary table so the comparison can be reproduced without refitting.

In [ ]:
print("leave-one-compound-out ...")
oof = np.full(len(y), np.nan); fold_feats = {}
for k, (tr, te) in enumerate(LeaveOneGroupOut().split(X, y, groups)):
    if len(np.unique(y[tr])) < 2:
        continue
    oof[te], feats = fit_predict(X.iloc[tr], y[tr], X.iloc[te])
    fold_feats[groups[te][0]] = list(feats)
    if (k + 1) % 25 == 0:
        print(f"   {k+1} folds done")
ok = ~np.isnan(oof)
tx_arm = arm_scores(oof[ok], arm_id[ok])

print("dose-only baseline ...")
oof_d = np.full(len(y), np.nan)
D = dose_ord.reshape(-1, 1)
for tr, te in LeaveOneGroupOut().split(D, y, groups):
    if len(np.unique(y[tr])) < 2:
        continue
    oof_d[te] = make_model(y[tr]).fit(D[tr], y[tr]).predict_proba(D[te])[:, 1]
okd = ~np.isnan(oof_d)
dose_arm_model = arm_scores(oof_d[okd], arm_id[okd])
dose_arm_raw = (pd.DataFrame({"arm": arm_id, "d": dose_ord})
                  .groupby("arm")["d"].first())

scores = (arm_truth
          .join(tx_arm.rename("score_transcriptomic"))
          .join(dose_arm_model.rename("score_dose_model"))
          .join(dose_arm_raw.rename("dose_level_ordinal"))).dropna()
scores.to_csv(WORK / "per_arm_scores.csv")
print(f"\nper-arm scores saved: {scores.shape}")
print(scores.head().to_string())

for c in ["score_transcriptomic", "score_dose_model", "dose_level_ordinal"]:
    print(f"{c:24s} AUC = {roc_auc_score(scores.y, scores[c]):.4f}  "
          f"AP = {average_precision_score(scores.y, scores[c]):.4f}")

## 5. DeLong test for correlated ROC curves

In [ ]:
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x)
    T = np.zeros(N, dtype=float); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T
    return T2

def _fast_delong(preds_sorted, m):
    n = preds_sorted.shape[1] - m
    k = preds_sorted.shape[0]
    pos, neg = preds_sorted[:, :m], preds_sorted[:, m:]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r] = _midrank(pos[r]); ty[r] = _midrank(neg[r]); tz[r] = _midrank(preds_sorted[r])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    return aucs, np.atleast_2d(cov)

def delong_test(y_true, score_a, score_b):
    """Two-sided DeLong test. Both scores must be on the SAME samples."""
    yv = np.asarray(y_true).astype(int)
    a = np.asarray(score_a, float); b = np.asarray(score_b, float)
    order = np.argsort(-yv)
    yv, a, b = yv[order], a[order], b[order]
    m = int(yv.sum())
    aucs, cov = _fast_delong(np.vstack([a, b]), m)
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    diff = aucs[0] - aucs[1]
    # Comparing a score with itself makes the covariance singular (var == 0).
    # inf * sign(0) is NaN, so the degenerate case is handled explicitly.
    if var <= 1e-15:
        z = 0.0 if abs(diff) < 1e-12 else float(np.inf) * np.sign(diff)
        p = 1.0 if abs(diff) < 1e-12 else 0.0
    else:
        z = diff / np.sqrt(var)
        p = 2 * (1 - stats.norm.cdf(abs(z)))
    return {"auc_a": float(aucs[0]), "auc_b": float(aucs[1]),
            "difference": float(diff), "se": float(np.sqrt(max(var, 0.0))),
            "z": float(z), "p_value": float(p)}

# self-test: a score against itself must give p = 1
_chk = delong_test(scores.y, scores.score_transcriptomic, scores.score_transcriptomic)
assert abs(_chk["difference"]) < 1e-9 and _chk["p_value"] > 0.99, "DeLong self-test failed"
_sk = roc_auc_score(scores.y, scores.score_transcriptomic)
assert abs(_chk["auc_a"] - _sk) < 1e-9, "DeLong AUC disagrees with sklearn"
print("DeLong self-tests passed\n")

DELONG = {}
DELONG["transcriptomic_vs_dose_model"] = delong_test(
    scores.y, scores.score_transcriptomic, scores.score_dose_model)
DELONG["transcriptomic_vs_dose_raw"] = delong_test(
    scores.y, scores.score_transcriptomic, scores.dose_level_ordinal)

for k, r in DELONG.items():
    print(f"{k}")
    print(f"   AUC {r['auc_a']:.4f} vs {r['auc_b']:.4f} | difference {r['difference']:+.4f} "
          f"(SE {r['se']:.4f}) | z = {r['z']:.2f} | p = {r['p_value']:.3g}\n")

## 6. Permutation test with 100 permutations

Arm labels are shuffled, keeping the arm structure. Observed and null both use the
light pipeline, so the comparison is internally consistent; using a different pipeline
for the null would make it an invalid reference.

In [ ]:
def loco_light(labels_vec):
    o = np.full(len(labels_vec), np.nan)
    for tr, te in LeaveOneGroupOut().split(X, labels_vec, groups):
        if len(np.unique(labels_vec[tr])) < 2:
            continue
        o[te] = fit_predict_fast(X.iloc[tr], labels_vec[tr], X.iloc[te])
    k = ~np.isnan(o)
    sc = arm_scores(o[k], arm_id[k])
    truth = pd.DataFrame({"arm": arm_id, "y": labels_vec}).groupby("arm").y.max().to_frame()
    j = truth.join(sc.rename("p"), how="inner").dropna()
    return roc_auc_score(j.y, j.p) if j.y.nunique() > 1 else np.nan

obs_light = loco_light(y)
print(f"observed (light pipeline): {obs_light:.4f}\n")

rng = np.random.default_rng(RANDOM_STATE)
null = []
for i in range(N_PERM):
    shuf = dict(zip(arm_truth.index, rng.permutation(arm_truth.y.values)))
    a = loco_light(np.array([shuf[x] for x in arm_id]))
    if not np.isnan(a):
        null.append(a)
    if (i + 1) % 10 == 0:
        print(f"   {i+1}/{N_PERM}  null mean so far {np.mean(null):.3f}")

null = np.array(null)
PERM = {"observed_light": float(obs_light),
        "null_mean": float(null.mean()), "null_sd": float(null.std()),
        "n_permutations": int(len(null)),
        "n_null_ge_observed": int((null >= obs_light).sum()),
        "p_value": float(((null >= obs_light).sum() + 1) / (len(null) + 1)),
        "z_vs_null": float((obs_light - null.mean()) / null.std())}
print("\n", PERM)
pd.DataFrame({"null_auc": null}).to_csv(WORK / "permutation_null.csv", index=False)

## 7. Sensitivity: does the signal survive without peroxisome proliferators?

Three PPARa agonists sit among the positive arms — clofibrate (2 positive arms),
fenofibrate (1) and gemfibrozil (3) — six of the 48 positives. Peroxisome
proliferation is a well-known rat-specific mechanism that does not translate to
humans, so a reviewer will ask whether the signature is driven by it.

WY-14643, the prototypical peroxisome proliferator, is in the dataset but **not**
among the admitted arms: it already causes injury by day 4, so it never enters the
pre-injury set. That is worth stating in the manuscript.

Two analyses are run. The **restricted evaluation** scores the existing predictions on
non-fibrate arms only: the fibrates were still in training, so this asks whether the
model works on other chemistry. The **full exclusion** removes them from training as
well and repeats the whole LOCO pass, which asks whether the signal depends on
fibrate biology at all. The second is the one to report; it costs another LOCO pass.

In [ ]:
FIBRATES = ["clofibrate", "fenofibrate", "gemfibrozil", "WY-14643"]
RUN_FULL_EXCLUSION = True     # set False to skip the extra LOCO pass (~1-2 h)

is_fib = np.array([c in FIBRATES for c in groups])
fib_arms = sorted(set(arm_id[is_fib]))
print(f"fibrate compounds present : {sorted(set(groups[is_fib]))}")
print(f"fibrate arms              : {len(fib_arms)}")
print(f"  positive among them     : "
      f"{int(arm_truth.loc[[a for a in fib_arms if a in arm_truth.index]].y.sum())}")

SENS = {}

# --- 7a. restricted evaluation (free: reuses the scores already computed) ---
keep = [a for a in scores.index if a not in set(fib_arms)]
sub = scores.loc[keep]
SENS["restricted_evaluation"] = {
    "n_arms": int(len(sub)), "n_positive": int(sub.y.sum()),
    "auc_transcriptomic": float(roc_auc_score(sub.y, sub.score_transcriptomic)),
    "auc_dose_raw": float(roc_auc_score(sub.y, sub.dose_level_ordinal)),
    "ap_transcriptomic": float(average_precision_score(sub.y, sub.score_transcriptomic))}
SENS["restricted_evaluation"]["delong_vs_dose_raw"] = delong_test(
    sub.y, sub.score_transcriptomic, sub.dose_level_ordinal)
print("\n7a. restricted evaluation (fibrates still in training)")
print(f"    arms {len(sub)}  positive {int(sub.y.sum())}")
print(f"    transcriptomic AUC {SENS['restricted_evaluation']['auc_transcriptomic']:.4f}"
      f"  vs dose {SENS['restricted_evaluation']['auc_dose_raw']:.4f}"
      f"  p = {SENS['restricted_evaluation']['delong_vs_dose_raw']['p_value']:.3g}")

# --- 7b. full exclusion: fibrates removed from training as well ---
if RUN_FULL_EXCLUSION:
    print("\n7b. full exclusion - repeating LOCO without fibrates ...")
    m = ~is_fib
    Xf, yf, gf, af = X[m], y[m], groups[m], arm_id[m]
    df_ord = dose_ord[m]
    print(f"    samples {len(yf)}  compounds {len(set(gf))}  arms {len(set(af))}")

    oof_f = np.full(len(yf), np.nan)
    for k, (tr, te) in enumerate(LeaveOneGroupOut().split(Xf, yf, gf)):
        if len(np.unique(yf[tr])) < 2:
            continue
        oof_f[te], _ = fit_predict(Xf.iloc[tr], yf[tr], Xf.iloc[te])
        if (k + 1) % 25 == 0:
            print(f"      {k+1} folds")
    kf = ~np.isnan(oof_f)

    oof_fd = np.full(len(yf), np.nan)
    Df = df_ord.reshape(-1, 1)
    for tr, te in LeaveOneGroupOut().split(Df, yf, gf):
        if len(np.unique(yf[tr])) < 2:
            continue
        oof_fd[te] = make_model(yf[tr]).fit(Df[tr], yf[tr]).predict_proba(Df[te])[:, 1]

    truth_f = pd.DataFrame({"arm": af, "y": yf}).groupby("arm").y.max().to_frame()
    sc_f = (truth_f
            .join(arm_scores(oof_f[kf], af[kf]).rename("score_transcriptomic"))
            .join(pd.DataFrame({"arm": af, "d": df_ord})
                    .groupby("arm")["d"].first().rename("dose_level_ordinal"))).dropna()
    sc_f.to_csv(WORK / "per_arm_scores_no_fibrates.csv")

    SENS["full_exclusion"] = {
        "n_arms": int(len(sc_f)), "n_positive": int(sc_f.y.sum()),
        "n_compounds": int(len(set(gf))),
        "auc_transcriptomic": float(roc_auc_score(sc_f.y, sc_f.score_transcriptomic)),
        "auc_dose_raw": float(roc_auc_score(sc_f.y, sc_f.dose_level_ordinal)),
        "ap_transcriptomic": float(average_precision_score(
            sc_f.y, sc_f.score_transcriptomic)),
        "delong_vs_dose_raw": delong_test(
            sc_f.y, sc_f.score_transcriptomic, sc_f.dose_level_ordinal)}
    r = SENS["full_exclusion"]
    print(f"\n    arms {r['n_arms']}  positive {r['n_positive']}  "
          f"compounds {r['n_compounds']}")
    print(f"    transcriptomic AUC {r['auc_transcriptomic']:.4f}"
          f"  vs dose {r['auc_dose_raw']:.4f}")
    print(f"    difference {r['delong_vs_dose_raw']['difference']:+.4f}"
          f"  DeLong p = {r['delong_vs_dose_raw']['p_value']:.3g}")
else:
    print("\n7b. skipped (RUN_FULL_EXCLUSION = False)")

print("\n" + "-" * 62)
print("full-cohort AUC for comparison: "
      f"{roc_auc_score(scores.y, scores.score_transcriptomic):.4f}")
print("A similar AUC after full exclusion means the signature is not a")
print("peroxisome-proliferation artefact. A large drop means it partly is,")
print("and that belongs in the limitations rather than being left unsaid.")

## 8. Save everything

In [ ]:
OUT = {"per_arm_auc": {c: float(roc_auc_score(scores.y, scores[c]))
                       for c in ["score_transcriptomic", "score_dose_model",
                                 "dose_level_ordinal"]},
       "per_arm_ap": {c: float(average_precision_score(scores.y, scores[c]))
                      for c in ["score_transcriptomic", "score_dose_model",
                                "dose_level_ordinal"]},
       "n_arms": int(len(scores)), "n_positive": int(scores.y.sum()),
       "delong": DELONG, "permutation": PERM, "fibrate_sensitivity": SENS}
(WORK / "step2b_results.json").write_text(json.dumps(OUT, indent=2))

print("=" * 66)
print("FOR THE MANUSCRIPT")
print("=" * 66)
d = DELONG["transcriptomic_vs_dose_raw"]
print(f"transcriptomic AUC  : {OUT['per_arm_auc']['score_transcriptomic']:.3f}")
print(f"dose-only AUC (raw) : {OUT['per_arm_auc']['dose_level_ordinal']:.3f}")
print(f"difference          : {d['difference']:+.3f} (SE {d['se']:.3f})")
print(f"DeLong              : z = {d['z']:.2f}, p = {d['p_value']:.3g}")
print(f"permutation         : null {PERM['null_mean']:.3f} +/- {PERM['null_sd']:.3f}, "
      f"p = {PERM['p_value']:.3g} ({PERM['n_permutations']} permutations)")
print(f"                      observed sits {PERM['z_vs_null']:.1f} SD above the null")

import shutil
arch = shutil.make_archive("/content/step2b_outputs", "zip", WORK,
                           base_dir=None)
print("\narchive:", arch)
try:
    from google.colab import files
    for f in ["per_arm_scores.csv", "step2b_results.json", "permutation_null.csv"]:
        files.download(str(WORK / f))
except Exception as e:
    print("files are in", WORK, "|", e)

## What to report

**Permutation.** Report the count, the null distribution and the p-value together:
"a permutation test with 100 arm-label permutations gave an empirical p of *p*, with the
observed AUC *z* SD above the null mean". If zero permutations reach the observed value
the p-value is 1/101 = 0.0099, and that bound should be stated as such rather than
written as "p < 0.05".

**DeLong.** Report the test against **raw dose** as the primary comparison — it is the
stricter and more interpretable baseline, since a fitted dose model can only approximate
what the ordinal dose already encodes. Give both AUCs, the difference with its standard
error, and the p-value.

**Per-arm scores.** `per_arm_scores.csv` belongs in the supplementary material. It lets a
reader reproduce every figure and both tests without refitting a single model, which is
the kind of thing reviewers of methodological papers notice.